# Конструирование алгоритмов `lscp`

In [1]:
import numpy as np
import networkx as nx
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox

from genesis.core import BestPointsBase, MetricBase, StateBase, StopCaseBase
from genesis.best_points import BestNodeHillClimbing
from genesis.states import FirstArrivalUnitState
from genesis.metrics import ArrivalTime, CoverIndex
from genesis.mclp import BestNodesKoptG
from genesis.tools import list_dict_concat
from genesis.utils import timing
from genesis.swiss_knife import CMAP_GrGldRd


In [2]:
# Загрузка графа и полигона границ
G = ox.load_graphml("tests/data/test_rng.ml")
area = gpd.read_file("tests/data/test_polygon.gpkg")

In [3]:
def iter_state_func(**kwds):
    '''
    Печать состояния расчета на итерации
    '''
    # print('Итерация:', kwds['i'])
    print(kwds)
    dynamic_nodes = kwds['dynamic_nodes']
    if isinstance(dynamic_nodes, dict):
        dynamic_nodes_id = list(dynamic_nodes.keys())
    else:
        dynamic_nodes_id = dynamic_nodes

    try:
        static_nodes = kwds['static_nodes']
        if isinstance(static_nodes, dict):
            static_nodes_id = list(static_nodes.keys())
        else:
            static_nodes_id = static_nodes
    except:
        static_nodes = None
        
    if not static_nodes is None:
        if isinstance(dynamic_nodes,list):
            start_nodes = dynamic_nodes+static_nodes
        elif isinstance(dynamic_nodes,dict):
            start_nodes = {**dynamic_nodes, **static_nodes}
    else:
        start_nodes = dynamic_nodes
    times, nearest = FirstArrivalUnitState()(env=G, points=start_nodes)
    times = {k:(1+int(t//5))*5 for k, t in times.items()}

    nx.set_node_attributes(G, pd.Series(nearest, dtype=str), 'nearest')
    nx.set_node_attributes(G, times, 'route_time')

    nds = ox.graph_to_gdfs(G, edges=False)
    fig, (ax1, ax2) = plt.subplots(1,2, figsize=(12,6))
    ox.plot_graph(G, edge_linewidth=0.2, node_size=0, show=False, ax=ax1)
    nds.plot(ax=ax1, markersize=2, column='nearest', cmap='Set3', legend=True)
    nds.loc[dynamic_nodes_id].plot(ax=ax1, markersize=25, color='red')
    if static_nodes:
        nds.loc[static_nodes_id].plot(ax=ax1, markersize=25, color='k')

    ox.plot_graph(G, edge_linewidth=0.2, node_size=0, show=False, ax=ax2)
    nds.plot(ax=ax2, markersize=2, column='route_time', cmap=CMAP_GrGldRd, legend=True)
    nds.loc[dynamic_nodes_id].plot(ax=ax2, markersize=25, color='red')
    if static_nodes:
        nds.loc[static_nodes_id].plot(ax=ax2, markersize=25, color='k')

    plt.show()

def iter_func(**kwds):
    'Функция вывода хода расчета'
    print(kwds)

# Разработка

In [31]:
from genesis.best_points import NodeMetric
from genesis.core import MetricBase, StateBase
from genesis.stop_cases import MoreEqualStopCase
from genesis.tools import get_dict_key


h = {}

class BestNodesGenesis(BestPointsBase):
    '''
    Расчет количества и оптимального размещения узлов для достижения целевой метрики.

    Выполняется методом `Генезис`.
    '''
    def __init__(self,
                 state_function: StateBase,
                 metric_function: MetricBase,
                 mclp_resolver: BestPointsBase,
                 stop_case: StopCaseBase,
                 appr_val: float = 0.95,
                 **kwargs):
        self.node_metric_func = NodeMetric(state_function,
                                           metric_function,
                                           appr_val,
                                           err_val=None,
                                           **kwargs)
        self.mclp_resolver = mclp_resolver
        self.stop_case = stop_case

        super().__init__(state_function, metric_function, **kwargs)

    def __call__(self,
                 env:nx.MultiDiGraph,
                 dynamic_nodes: dict,
                 static_nodes: dict = None,
                 area: pd.Series = None,
                 **kwargs):
        '''
        Запуск работы алгоритма

        ## Аргументы
        `env`:nx.MultiDiGraph
            Граф улично-дорожной сети
        `dynamic_nodes`: list|dict
            Список стартовых узлов графа в которых размещены
            подразделения оптимальные места которых следует определить.
        `static_nodes`: list|dict = None
            Список стартовых узлов графа в которых размещены
            подразделения изменять размещение которых не следует.
        `area`: pd.Series = None
            Маска узлов графа. Значениями True отмечены узлы графа - цели расчета леса Вороного.
            Если не указана, расчет производится для всех узлов графа.
        '''

        best_dynamic_nodes = dynamic_nodes

        # Опускаем флаг прекращения расчетов
        stop_flag = False

        # Итерации
        iteration = 0
        while not stop_flag:
            # 1. Расчет оптимального размещения подразделений
            best_dynamic_nodes, best_metric = self.mclp_resolver(env=env,
                                            dynamic_nodes=best_dynamic_nodes,
                                            static_nodes=static_nodes,
                                            area=area,
                                            **kwargs)

            # 2. Если достигнута цель расчета, выходим из цикла
            if self.stop_case(value=best_metric,
                            iteration=iteration,
                            best_metric=best_metric,
                            dynamic_nodes=best_dynamic_nodes,
                            static_nodes=static_nodes):
                return best_dynamic_nodes, best_metric

            # ============================================================================================
            # 3. Если нет - создаем новое подразделение
            # 3.1. Получение суммарного списка (или словаря) узлов для расчета состояния и метрик
            start_nodes = list_dict_concat(dynamic_nodes, static_nodes)

            # 3.2. Расчет состояния
            times, nearest = self.state_function(env=env, points=start_nodes, **kwargs)
            print(times)
            h['times'] = times
            h['nearest'] = nearest

            # 3.2. Получаем подразделение с худшим расположением
            # 3.2.1. Расчет метрик для каждого из подразделений и выбор худшего
            # nodes_metrics = {}
            worst_node, worst_metric, worst_nodes_part = None, None, None
            for point in nearest.unique():

                # # Проверяем является ли точка одной из динамических
                # if point in static_nodes.values():

                # times_part = pd.Series([t for n,t in zip(nearest, times) if n==point], dtype=times.dtype)
                times_part = pd.Series({tk:tv for n,tk,tv in zip(nearest, times.to_dict().keys(), times.to_dict().values()) if n==point}, dtype=h['times'].dtype)
                # print(times_part)
                # print(start_point)
                metric_part = self.metric_function(times_part)
                if worst_node is None:
                    worst_node, worst_metric = point, metric_part
                    worst_nodes_part = list(times_part.keys())
                else:
                    # Если худшая из найденных метрик лучше чем метрика для текущего узла, 
                    # выбираем метрику для текущего узла
                    if self.metric_function.compare(worst_metric, metric_part) == worst_metric:
                        worst_node, worst_metric = point, metric_part
                        worst_nodes_part = list(times_part.keys())
            
            worst_node = get_dict_key(start_nodes, worst_node)   # start_nodes worst_node
            h['worst_node'] = worst_node
            h['worst_nodes_part'] = worst_nodes_part

            # !? Способ выбора сделать опциональным
            # Рядом с подразделением с худшим расположением создаем узел-кандидат
            best_second_node, best_part_metric = None, None
            for node in env[worst_node]:

                print(worst_node, node)

                # Определяем подграф зоны обслуживания для узла dynamic_node
                node_part_env = nx.subgraph(env, worst_nodes_part)

                # Определяем метрику для узла-кандидата
                # node_metric = self.node_metric_func(env=env, node=start_node, area=area, **kwargs)
                times, _ = self.state_function(env=node_part_env, points=[worst_node, node], area=area, **kwargs)
                node_part_metric = self.metric_function(times)

                # Проверяем какой из узлов лучше
                # if best_second_node is None:
                #     best_second_node, best_part_metric = node, node_part_metric
                # else:
                # !? Здесь сейчас - луший узел. Но возможно стоит рассмотреть худший
                if self.metric_function.compare(best_part_metric, node_part_metric) == node_part_metric:
                    best_second_node, best_part_metric = node, node_part_metric

            # Добавление нового узла в словарь динамических узлов:
            best_dynamic_nodes[best_second_node] = str(iteration)
            print(best_dynamic_nodes)

            iteration += 1
            
            


class ItersStopCase(StopCaseBase):
    '''
    Критерий остановки по условию, что на протяжении нескольких итераций
    значения равны, т.е. не изменяются
    '''
    def __init__(self, max_iter_count=2):
        self.max_iter_count = max_iter_count

    def __call__(self, **kwargs):
        print(kwargs)
        iters_count = kwargs['iteration']
        
        return iters_count>self.max_iter_count



# Определение маски расчета
nodes = ox.graph_to_gdfs(G, edges=False)
points_mask = nodes.within(area.iloc[0].geometry)


nodes = list(G.nodes())
existed_units = {nodes[100]:'A', 
                nodes[2000]:'B', 
                }
new_units = {nodes[3000]:'c',
             nodes[4000]:'d'}



BNHC = BestNodeHillClimbing(state_function = FirstArrivalUnitState(),
                            metric_function = ArrivalTime())

BNG = BestNodesKoptG(state_function = FirstArrivalUnitState(),
                    metric_function = ArrivalTime(),
                    best_point_function = BNHC,
                    iterations = 5,
                    )

stop_case = ItersStopCase(max_iter_count = 2)


genesis = BestNodesGenesis(FirstArrivalUnitState(), ArrivalTime(), BNG, stop_case)

genesis(env=G, dynamic_nodes = new_units, static_nodes = existed_units)


{'value': 6.984608999416008, 'iteration': 0, 'best_metric': 6.984608999416008, 'dynamic_nodes': {4116059950: 'c', 5116722296: 'd'}, 'static_nodes': {1296599762: 'A', 2042076750: 'B'}}
2760591335      1.000000
4901942432      1.000000
1296599762      1.000000
2042076750      1.000000
2042076753      1.012382
                 ...    
10816268037    22.610905
10816268038    22.970161
10816268039    23.364625
10816268040    23.609989
10816268041    23.958373
Name: times, Length: 5504, dtype: float64
2042076750 2042076753


NodeNotFound: Node 2042076753 not found in graph

In [32]:
G[2042076753]

AdjacencyView({2042076766: {0: {'osmid': 193689157, 'highway': 'service', 'oneway': False, 'reversed': False, 'length': 28.616, 'travel_time': 0.057232}}, 2042076750: {0: {'osmid': 193689157, 'highway': 'service', 'oneway': False, 'reversed': True, 'length': 6.191, 'travel_time': 0.012381999999999999}}})

In [9]:
new_units, existed_units

({2760591335: 'c', 4901942432: 'd'}, {1296599762: 'A', 2042076750: 'B'})

In [12]:
h

{'worst_node': 2042076750,
 'worst_nodes_part': [0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  120,
  121,
  122,
  123,
  124,
  125,
  126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
  136,
  137,
  138,
  139,
  140,
  141,
  142,
  143,
  144,
  145,
  146,
  147,
  148,
  149,
  150,
  151,